# Principal Component Analysis (PCA)

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## A Different Kind of Unsupervised Learning

K-Means and DBSCAN found **groups** in the data. PCA does something different — it finds the **most important directions** in the data and compresses it.

> *Find the directions that capture the most variance, then represent the data using fewer dimensions without losing much information.*

---

## Why Compress?

Three reasons:

**Visualization** — you can't plot 7D data. Project onto 2 principal components and you get a 2D plot capturing as much structure as possible.

**Noise reduction** — later components often capture noise. Dropping them can improve downstream model performance.

**Curse of dimensionality** — distance-based algorithms (KNN, SVM, K-Means) work better in fewer dimensions.

---

## How PCA Works

1. **Center the data** — subtract the mean from each feature
2. **Compute the covariance matrix** — captures how features vary together
3. **Eigendecompose** — finds the principal directions (eigenvectors) and how much variance each captures (eigenvalues)
4. **Sort by variance** — largest eigenvalue = most important direction
5. **Project** — multiply centered data by the top K eigenvectors

No gradient descent. No iterations. One closed-form matrix computation.

---

## Explained Variance

Each principal component's eigenvalue tells you what fraction of total variance it explains. If the first 2 components explain 85% of variance, a 2D projection preserves 85% of the data's information.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, '../../src')
from football_ml.unsupervised_learning.pca import PCA

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. PCA on Toy Data — The Intuition

In [ ]:
rng = np.random.default_rng(0)
t = rng.uniform(-3, 3, 100)
X_toy = np.c_[
    t + rng.normal(0, 0.2, 100),
    t + rng.normal(0, 0.2, 100)
]

pca_toy = PCA(n_components=2).fit(X_toy)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original data with principal component arrows
axes[0].scatter(X_toy[:, 0], X_toy[:, 1], alpha=0.5, color='#4878CF', s=20)
for i, (comp, var) in enumerate(zip(pca_toy.components_, pca_toy.explained_variance_)):
    scale = np.sqrt(var) * 2
    axes[0].annotate('', xy=pca_toy.mean_ + scale * comp,
                     xytext=pca_toy.mean_,
                     arrowprops=dict(arrowstyle='->', color=['#E87272','#2ecc71'][i], lw=2.5))
    axes[0].text(*(pca_toy.mean_ + scale * comp * 1.1),
                 f'PC{i+1}\n({pca_toy.explained_variance_ratio_[i]:.0%})',
                 fontsize=9, ha='center')
axes[0].set_title('Original Data + Principal Components', fontsize=11, fontweight='bold')
axes[0].set_aspect('equal')

# Projected data (1D)
X_proj = pca_toy.transform(X_toy)
axes[1].scatter(X_proj[:, 0], np.zeros(len(X_proj)),
                alpha=0.5, color='#E87272', s=20)
axes[1].set_yticks([])
axes[1].set_xlabel('PC1 (first principal component)', fontsize=11)
axes[1].set_title(f'Projected onto PC1\n({pca_toy.explained_variance_ratio_[0]:.0%} variance retained)',
                  fontsize=11, fontweight='bold')

plt.suptitle('PCA — Finding the Most Important Direction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The red arrow is PC1 — the direction of maximum variance. The green arrow is PC2 — perpendicular to PC1, capturing the remaining variance. Projecting onto PC1 alone retains most of the data's structure.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win']  = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

# Always scale before PCA — variance is meaningless if features have
# different units/scales
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

print(f'Shape: {X_sc.shape}')

---
## 4. Fit PCA — All Components

In [ ]:
pca_full = PCA().fit(X_sc)

print('Explained variance per component:')
for i, (var, ratio, cum) in enumerate(zip(
    pca_full.explained_variance_,
    pca_full.explained_variance_ratio_,
    pca_full.cumulative_variance_ratio_
)):
    print(f'  PC{i+1}: variance={var:.3f}  ratio={ratio:.1%}  cumulative={cum:.1%}')

n_95 = pca_full.n_components_for_variance(0.95)
print(f'\nComponents needed for 95% variance: {n_95}')

---
## 5. Scree Plot — How Many Components?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
n_comp = len(pca_full.explained_variance_ratio_)
x = np.arange(1, n_comp + 1)

# Individual variance per component
axes[0].bar(x, pca_full.explained_variance_ratio_ * 100,
            color='#4878CF', edgecolor='white')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance (%)', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)

# Cumulative variance
axes[1].plot(x, pca_full.cumulative_variance_ratio_ * 100,
             'o-', color='#E87272', linewidth=2)
axes[1].axhline(95, color='gray', linestyle='--', linewidth=1.2, label='95% threshold')
axes[1].axvline(n_95, color='black', linestyle='--', linewidth=1.2,
                label=f'{n_95} components')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Variance (%)', fontsize=12)
axes[1].set_title('Cumulative Explained Variance', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xticks(x)

plt.tight_layout()
plt.show()

---
## 6. 2D Projection — Visualising 7D Data

In [ ]:
pca_2d = PCA(n_components=2).fit(X_sc)
X_2d = pca_2d.transform(X_sc)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color, name in [(0, '#E87272', 'Draw/Away'), (1, '#4878CF', 'Home Win')]:
    mask = y == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               color=color, alpha=0.3, s=8, label=name)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('Football Matches Projected onto First 2 Principal Components',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Total variance retained in 2D: {pca_2d.cumulative_variance_ratio_[-1]:.1%}')

This 2D plot is the best possible 2D representation of the original 7D data. The overlap between classes confirms what we already knew — football outcomes are hard to separate cleanly from these features.

---
## 7. Component Loadings — What Does Each PC Mean?

In [ ]:
loadings = pd.DataFrame(
    pca_full.components_[:4].T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(4)]
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 9})
ax.set_title('PCA Loadings — Feature Contributions to Each PC',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Each cell shows how much a feature contributes to a principal component. Large positive values (red) mean that feature pushes the PC score up; negative (blue) pushes it down. This lets you interpret what each PC actually represents — e.g. PC1 might represent "overall attacking strength" if both home and away rolling goals load highly.

---
## 8. Reconstruction — Information Loss

In [ ]:
n_values = list(range(1, 8))
errors = []
for n in n_values:
    m = PCA(n_components=n).fit(X_sc)
    errors.append(m.reconstruction_error(X_sc))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_values, errors, 'o-', color='#4878CF', linewidth=2)
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Reconstruction Error (MSE)', fontsize=12)
ax.set_title('Information Loss vs Compression', fontsize=12, fontweight='bold')
ax.set_xticks(n_values)
plt.tight_layout()
plt.show()

As we keep more components, reconstruction error drops toward zero. With all 7 components the reconstruction is perfect — no information lost. With fewer components we trade information for compression.

---
## 9. PCA as Preprocessing — Does It Help Classification?

In [ ]:
from sklearn.model_selection import train_test_split
from football_ml.supervised_learning.logistic_regression import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y, test_size=0.2, random_state=SEED, stratify=y
)

results = []
for n in [2, 3, 4, 5, 6, 7]:
    pca_n = PCA(n_components=n).fit(X_train)
    X_tr_pca = pca_n.transform(X_train)
    X_te_pca = pca_n.transform(X_test)

    lr = LogisticRegression(learning_rate=0.1, n_epochs=500, random_state=SEED)
    lr.fit(X_tr_pca, y_train)
    results.append({'n_components': n, 'accuracy': lr.score(X_te_pca, y_test)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

---
## 10. Discussion

### What PCA does well
- **Visualization**: turns high-dimensional data into something you can see
- **Noise reduction**: dropping low-variance components removes noise
- **Speed**: downstream models train faster on fewer features
- **Closed-form**: no hyperparameter tuning beyond n_components, no gradient descent
- **Interpretability**: loadings reveal what the principal components represent

### What it struggles with
1. **Linear only**: PCA finds linear combinations of features. Non-linear structure (like concentric rings) is missed
2. **Unsupervised**: PCA doesn't know about labels — the directions it finds maximize variance, not class separation
3. **Interpretability loss**: the new features (PCs) are combinations of original features — harder to explain to stakeholders
4. **Must scale first**: mandatory, or high-variance features dominate entirely

### For football

With only 7 features, PCA compression is modest. Its real value here is visualization — the 2D projection lets us see the overall structure of match data. The overlap between classes confirms that these 7 features don't cleanly separate home wins from other outcomes.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Dimensionality reduction |
| **Approach** | Eigendecomposition of covariance matrix |
| **Output** | Lower-dimensional coordinates |
| **Training** | Closed-form — no iterations |
| **Key parameter** | n_components |
| **Evaluation** | Explained variance ratio, reconstruction error |
| **Must scale first** | Yes — always |
| **Key use cases** | Visualization, noise reduction, preprocessing |
